In [14]:
import torch
import torch.nn as nn
import math
import pickle
from collections import Counter, OrderedDict


# 1. VOCABULARY

class Vocabulary:
    """Build and manage vocabulary for a language"""
    def __init__(self, max_vocab_size=10000):
        self.word2idx = {'<PAD>': 0, '<UNK>': 1, '<SOS>': 2, '<EOS>': 3}
        self.idx2word = {0: '<PAD>', 1: '<UNK>', 2: '<SOS>', 3: '<EOS>'}
        self.word_count = Counter()
        self.max_vocab_size = max_vocab_size

    def build_vocab(self, sentences):
        """Build vocabulary from sentences"""
        for sentence in sentences:
            words = sentence.split()
            self.word_count.update(words)

        idx = 4
        for word, count in self.word_count.most_common(self.max_vocab_size - 4):
            self.word2idx[word] = idx
            self.idx2word[idx] = word
            idx += 1

    def encode(self, sentence):
        """Convert sentence to indices"""
        return [self.word2idx.get(word, 1) for word in sentence.split()]

    def decode(self, indices):
        """Convert indices back to sentence"""
        words = [self.idx2word.get(idx, '<UNK>') for idx in indices]
        return ' '.join(words)

    def __len__(self):
        return len(self.word2idx)



# 2. THE TRANSFORMER ARCHITECTURE

class PositionalEncoding(nn.Module):
    """Add positional encoding to embeddings"""
    def __init__(self, d_model, max_seq_len=5000, dropout=0.3):
        super().__init__()
        self.dropout = nn.Dropout(p=dropout)

        pe = torch.zeros(max_seq_len, d_model)
        position = torch.arange(0, max_seq_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * -(math.log(10000.0) / d_model))

        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)

        self.register_buffer('pe', pe.unsqueeze(0))

    def forward(self, x):
        x = x + self.pe[:, :x.size(1), :]
        return self.dropout(x)

class MultiHeadAttention(nn.Module):
    """Multi-head attention mechanism"""
    def __init__(self, d_model, num_heads, dropout=0.3):
        super().__init__()
        assert d_model % num_heads == 0

        self.d_model = d_model
        self.num_heads = num_heads
        self.d_k = d_model // num_heads

        self.W_q = nn.Linear(d_model, d_model)
        self.W_k = nn.Linear(d_model, d_model)
        self.W_v = nn.Linear(d_model, d_model)
        self.W_o = nn.Linear(d_model, d_model)

        self.dropout = nn.Dropout(dropout)

    def forward(self, Q, K, V, mask=None):
        batch_size = Q.shape[0]

        Q = self.W_q(Q).view(batch_size, -1, self.num_heads, self.d_k).transpose(1, 2)
        K = self.W_k(K).view(batch_size, -1, self.num_heads, self.d_k).transpose(1, 2)
        V = self.W_v(V).view(batch_size, -1, self.num_heads, self.d_k).transpose(1, 2)

        scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(self.d_k)

        if mask is not None:
            scores = scores.masked_fill(mask == 0, -1e9)

        attention_weights = torch.softmax(scores, dim=-1)
        attention_weights = self.dropout(attention_weights)

        context = torch.matmul(attention_weights, V)
        context = context.transpose(1, 2).contiguous()
        context = context.view(batch_size, -1, self.d_model)

        output = self.W_o(context)
        return output, attention_weights

class FeedForwardNetwork(nn.Module):
    """Position-wise feed-forward network"""
    def __init__(self, d_model, d_ff, dropout=0.3):
        super().__init__()
        self.linear1 = nn.Linear(d_model, d_ff)
        self.linear2 = nn.Linear(d_ff, d_model)
        self.dropout = nn.Dropout(dropout)
        self.activation = nn.ReLU()

    def forward(self, x):
        return self.linear2(self.dropout(self.activation(self.linear1(x))))

class TransformerEncoderLayer(nn.Module):
    """Single transformer encoder layer"""
    def __init__(self, d_model, num_heads, d_ff, dropout=0.3):
        super().__init__()
        self.attention = MultiHeadAttention(d_model, num_heads, dropout)
        self.ffn = FeedForwardNetwork(d_model, d_ff, dropout)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, mask=None):
        attn_output, _ = self.attention(x, x, x, mask)
        x = self.norm1(x + self.dropout(attn_output))
        ffn_output = self.ffn(x)
        x = self.norm2(x + self.dropout(ffn_output))
        return x

class TransformerDecoderLayer(nn.Module):
    """Single transformer decoder layer"""
    def __init__(self, d_model, num_heads, d_ff, dropout=0.3):
        super().__init__()
        self.self_attention = MultiHeadAttention(d_model, num_heads, dropout)
        self.cross_attention = MultiHeadAttention(d_model, num_heads, dropout)
        self.ffn = FeedForwardNetwork(d_model, d_ff, dropout)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.norm3 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, encoder_output, src_mask=None, tgt_mask=None):
        self_attn_output, _ = self.self_attention(x, x, x, tgt_mask)
        x = self.norm1(x + self.dropout(self_attn_output))
        cross_attn_output, _ = self.cross_attention(x, encoder_output, encoder_output, src_mask)
        x = self.norm2(x + self.dropout(cross_attn_output))
        ffn_output = self.ffn(x)
        x = self.norm3(x + self.dropout(ffn_output))
        return x

class TransformerNMT(nn.Module):
    """Complete Transformer for Neural Machine Translation"""
    def __init__(self, src_vocab_size, tgt_vocab_size, d_model=512, num_heads=4,
                 num_layers=3, d_ff=1024, max_seq_len=50, dropout=0.3):
        super().__init__()
        self.src_embedding = nn.Embedding(src_vocab_size, d_model, padding_idx=0)
        self.tgt_embedding = nn.Embedding(tgt_vocab_size, d_model, padding_idx=0)
        self.positional_encoding = PositionalEncoding(d_model, max_seq_len, dropout)

        self.encoder_layers = nn.ModuleList([
            TransformerEncoderLayer(d_model, num_heads, d_ff, dropout) for _ in range(num_layers)
        ])
        self.decoder_layers = nn.ModuleList([
            TransformerDecoderLayer(d_model, num_heads, d_ff, dropout) for _ in range(num_layers)
        ])

        self.fc_out = nn.Linear(d_model, tgt_vocab_size)
        self.dropout = nn.Dropout(dropout)

    def create_padding_mask(self, seq):
        return (seq != 0).unsqueeze(1).unsqueeze(2)

    def create_causal_mask(self, seq_len, device):
        mask = torch.triu(torch.ones(seq_len, seq_len, device=device), diagonal=1)
        return (mask == 0).unsqueeze(0).unsqueeze(0)

    def encode(self, src, src_mask=None):
        x = self.dropout(self.positional_encoding(self.src_embedding(src)))
        for layer in self.encoder_layers:
            x = layer(x, src_mask)
        return x

    def decode(self, tgt, encoder_output, src_mask=None, tgt_mask=None):
        x = self.dropout(self.positional_encoding(self.tgt_embedding(tgt)))
        for layer in self.decoder_layers:
            x = layer(x, encoder_output, src_mask, tgt_mask)
        return x

    def forward(self, src, tgt):
        src_mask = self.create_padding_mask(src)
        tgt_mask = self.create_padding_mask(tgt)
        tgt_mask = tgt_mask & self.create_causal_mask(tgt.size(1), tgt.device)

        encoder_output = self.encode(src, src_mask)
        decoder_output = self.decode(tgt, encoder_output, src_mask, tgt_mask)
        logits = self.fc_out(decoder_output)
        return logits



# 3. THE TRANSLATE FUNCTION

@torch.no_grad()
def translate(model, src_sentence, src_vocab, tgt_vocab, device, max_len=50):
    """Translate a single sentence using autoregressive decoding"""
    model.eval()

    # 1. Prepare Source
    src_ids = [2] + src_vocab.encode(src_sentence) + [3]
    src_tensor = torch.tensor(src_ids).unsqueeze(0).to(device)

    # 2. Create source mask and encode
    src_mask = model.create_padding_mask(src_tensor)
    encoder_output = model.encode(src_tensor, src_mask)

    # 3. Start target sequence with <SOS>
    tgt_ids = torch.tensor([[2]], device=device)

    # 4. Autoregressive loop
    for _ in range(max_len):
        tgt_mask = model.create_padding_mask(tgt_ids)
        tgt_mask = tgt_mask & model.create_causal_mask(tgt_ids.size(1), device)

        decoder_output = model.decode(tgt_ids, encoder_output, src_mask, tgt_mask)
        logits = model.fc_out(decoder_output)

        next_token = logits[0, -1, :].argmax(dim=-1).item()
        tgt_ids = torch.cat([tgt_ids, torch.tensor([[next_token]], device=device)], dim=1)

        if next_token == 3:  # Stop if <EOS> is generated
            break

    # 5. Decode to text
    tgt_sentence = tgt_vocab.decode(tgt_ids[0, 1:].cpu().numpy())
    return tgt_sentence.replace('<EOS>', '').strip()



# 4. MAIN EXECUTION

def main():
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"Using device: {device}")

    # 1. Load the Dictionaries
    print("Loading vocabularies...")
    try:
        with open('src_vocab.pkl', 'rb') as f:
            src_vocab = pickle.load(f)
        with open('tgt_vocab.pkl', 'rb') as f:
            tgt_vocab = pickle.load(f)
    except FileNotFoundError:
        print("ERROR: Could not find vocabulary files. Make sure src_vocab.pkl and tgt_vocab.pkl are in the same folder.")
        return

    # 2. The empty Model
    print("Initializing model architecture...")
    model = TransformerNMT(
        src_vocab_size=len(src_vocab),
        tgt_vocab_size=len(tgt_vocab),
        d_model=512,
        num_heads=4,
        num_layers=3,
        d_ff=1024,
        max_seq_len=50,
        dropout=0.3
    )

    # 3. Load the Weights
    print("Loading trained weights...")
    try:
        state_dict = torch.load('best_model-4.pt', map_location=device, weights_only=False)

        # Remove DataParallel 'module.' prefix if it exists
        new_state_dict = OrderedDict()
        for k, v in state_dict.items():
            name = k[7:] if k.startswith('module.') else k
            new_state_dict[name] = v

        model.load_state_dict(new_state_dict)
    except FileNotFoundError:
        print("ERROR: Could not find best_model-4.pt. Make sure it is in the same folder.")
        return

    model = model.to(device)
    model.eval()
    print("Model loaded successfully!\n")

    # 4. Test Translations
    print("="*60)
    print("TESTING INFERENCE")
    print("="*60)

    sentences_to_translate = [
        "Hello, how are you?",
        "What is your name?",
        "I am good"
    ]

    for sent in sentences_to_translate:
        translation = translate(model, sent, src_vocab, tgt_vocab, device)
        print(f"English: {sent}")
        print(f"Hindi:   {translation}\n")


if __name__ == "__main__":
    main()

Using device: cpu
Loading vocabularies...
Initializing model architecture...
Loading trained weights...
Model loaded successfully!

TESTING INFERENCE
English: Hello, how are you?
Hindi:   हैलो, तुम कैसे हो?

English: What is your name?
Hindi:   आपका नाम क्या है?

English: I am good
Hindi:   मैं अच्छा हूँ.

